# Human grading: run agent on golden cases, grade each trajectory

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import yaml

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from agent.context import load_prompt
from agent.judge import JUDGE_PROMPT, parse_json
from agent.llm import LLM
from agent.loop import investigate
from agent.schema import CRITERIA, Judgement
from agent.tools import LogTools, tool_purposes
from agent.trajectory import EventLog, Trajectory
from logstore.db import connect

GOLDEN_PATH = ROOT / "labeling" / "golden_cases.yaml"
GRADES_PATH = ROOT / "labeling" / "human_grades.csv"
CACHE_TAG = "golden"

cases = yaml.safe_load(GOLDEN_PATH.read_text(encoding="utf-8"))
print(f"{len(cases)} golden cases: {[c['id'] for c in cases]}")

## Run agent on all cases in parallel

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def run_one(case):
    conn = connect()
    tools = LogTools(conn)
    found = tools.logs.find_by_line(case["log_line"])
    assert len(found) == 1, f"{case['id']}: {len(found)} logs match"
    log = found[0]
    tools.visible = log.ts
    request_id = investigate(log, tools, LLM(), EventLog(), cache_tag=CACHE_TAG)
    return case["id"], request_id


with ThreadPoolExecutor(max_workers=5) as pool:
    futures = {pool.submit(run_one, c): c["id"] for c in cases}
    results = {}
    for future in as_completed(futures):
        case_id, request_id = future.result()
        results[case_id] = request_id
        t = Trajectory(request_id)
        print(f"{case_id}: {request_id}  status={t.status}  rounds={t.totals['rounds']}")

print(f"\ndone: {len(results)} investigations")

## The judge prompt
Read this once. Each case below shows the exact input the judge gets. Paste your JSON grade in the cell after it.

In [ ]:
print(f"{load_prompt(JUDGE_PROMPT)} \n\n # THE AGENT'S TOOLS\n{tool_purposes()}\n\n")

In [ ]:
def print_trajectory(case_id: str):
    print(f"# THE TRAJECTORY\n{Trajectory(results[case_id]).text()}")

## c01

In [ ]:
print_trajectory('c01')

In [ ]:
c01 = """paste JSON here"""

## c02

In [ ]:
print_trajectory('c02')

In [ ]:
c02 = """paste JSON here"""

## c03

In [ ]:
print_trajectory('c03')

In [ ]:
c03 = """paste JSON here"""

## c05

In [ ]:
print_trajectory('c05')

In [ ]:
c05 = """paste JSON here"""

## c06

In [ ]:
print_trajectory('c06')

In [ ]:
c06 = """paste JSON here"""

## c07

In [ ]:
print_trajectory('c07')

In [ ]:
c07 = """paste JSON here"""

## c09

In [ ]:
print_trajectory('c09')

In [ ]:
c09 = """paste JSON here"""

## c10

In [ ]:
print_trajectory('c10')

In [ ]:
c10 = """paste JSON here"""

## c12

In [ ]:
print_trajectory('c12')

In [ ]:
c12 = """paste JSON here"""

## c17

In [ ]:
print_trajectory('c17')

In [ ]:
c17 = """paste JSON here"""

## Validate and save

In [ ]:
rows = []
for case_id, raw in [("c01", c01), ("c02", c02), ("c03", c03), ("c05", c05), ("c06", c06),
                      ("c07", c07), ("c09", c09), ("c10", c10), ("c12", c12), ("c17", c17)]:
    if raw.strip() == "paste JSON here":
        continue
    judgement = Judgement.model_validate(parse_json(raw))
    row = {"case_id": case_id, "request_id": results[case_id]}
    for criterion in CRITERIA:
        row[criterion] = getattr(judgement, criterion).result
    rows.append(row)
    print(f"{case_id}: {', '.join(c + '=' + row[c] for c in CRITERIA)}")

if rows:
    pd.DataFrame(rows).to_csv(GRADES_PATH, index=False)
    print(f"\nsaved {len(rows)} rows to {GRADES_PATH}")
else:
    print("no grades to save yet")